# Treinamento, teste e linha de base

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_08/02_treinamento_teste_e_linha_de_base.ipynb)

O conjunto de treinamento ensina parâmetros; o teste estima desempenho em casos
não usados no ajuste. A divisão deve ocorrer antes de decisões guiadas pelo
resultado. Estratificar preserva aproximadamente a proporção da classe-alvo.

![Os dados anotados se dividem em treino e teste, e o teste permanece reservado.](imagens/02_treino_teste.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_08'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('sklearn', 'scikit-learn>=1.5,<2')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

dados = pd.read_csv("dados/documentos_classificacao.csv")
treino, teste = train_test_split(dados, test_size=.25, random_state=42, stratify=dados["classe_alvo"])
print("Treino:", len(treino), "| Teste reservado:", len(teste))
display(pd.crosstab(index=["treino"], columns=treino["classe_alvo"], normalize="index").round(3))
display(pd.crosstab(index=["teste"], columns=teste["classe_alvo"], normalize="index").round(3))

## 2. Linha de base antes do modelo

Uma linha de base responde: o método supera uma regra trivial? Usaremos a classe
mais frequente. Como as classes são quase equilibradas, a acurácia da maioria deve
ficar perto de 50%. Com **desbalanceamento**, ela pode parecer alta e ainda ignorar
inteiramente a classe de interesse.

In [ ]:
linha_base = DummyClassifier(strategy="most_frequent")
linha_base.fit([[0]] * len(treino), treino["classe_alvo"])
previsto_base = linha_base.predict([[0]] * len(teste))

def metricas(y, previsto):
    return pd.Series({
        "acurácia": accuracy_score(y, previsto),
        "precisão": precision_score(y, previsto, pos_label="relevante", zero_division=0),
        "revocação": recall_score(y, previsto, pos_label="relevante", zero_division=0),
        "F1": f1_score(y, previsto, pos_label="relevante", zero_division=0),
    })

metricas_base = metricas(teste["classe_alvo"], previsto_base)
display(metricas_base.to_frame("linha de base").T.round(3))

### U08-A03 — Atividade — proteger o teste

Explique por que não devemos escolher palavras, parâmetros ou regras após olhar os
erros do teste repetidamente. Compare uma divisão estratificada e outra sem
estratificação; registre proporções, semente e qual versão será preservada.

**Dinâmica (25 minutos):** individual na execução; dupla na justificativa. Não
escolha a divisão que produz a maior métrica.

**Resposta:** escreva aqui.

## Passagem

Fixados teste e linha de base, podemos comparar modelos. A avaliação seguinte não
se limita a um número: separa tipos de erro e grupos afetados.